# LAB 1 — OpenSearch Local com Docker Desktop

**Curso:** MBA RAG & CAG Aplicados a Direito e Segurança Pública  
**Aula:** 1 — Fundamentos  
**Duração estimada:** 25 minutos  
**Ambiente:** Jupyter Local no VS Code

---

## Objetivos de Aprendizagem
1. Subir o **OpenSearch 3.x** e **Dashboards** via `docker compose` na sua máquina
2. Conectar o notebook Python ao OpenSearch local via `opensearch-py`
3. Criar um índice jurídico com mapeamento de campos adequado
4. Indexar documentos jurídicos e executar buscas por texto completo
5. Validar o ambiente com um checklist automatizado

## Arquitetura do Lab
```
Notebook Jupyter (VS Code)
    |
    | HTTP (localhost)
    |
    v
Docker Desktop
    |
    +-- opensearch-node1:9200  (motor de busca)
    +-- opensearch-dashboards:5601  (interface web)
```

> **PRÉ-REQUISITO:** Docker Desktop instalado e o OpenSearch rodando.  
> Consulte o `ROTEIRO_INSTALACAO_FERRAMENTAS.md` (seções 1 e 5) se ainda não fez.

---
## PARTE 1 — SUBIR O OPENSEARCH (execute no terminal da sua maquina)

### Passo A — Criar o arquivo `docker-compose.yml`

Em um terminal, execute:

```bash
mkdir -p ~/mba-rag/infra
cd ~/mba-rag/infra
```

Crie o arquivo `docker-compose.yml` com o conteúdo abaixo:

```yaml
version: '3'
services:
  opensearch-node1:
    image: opensearchproject/opensearch:3.6.0
    container_name: opensearch-node1
    environment:
      - cluster.name=opensearch-cluster-rag
      - node.name=opensearch-node1
      - discovery.type=single-node
      - bootstrap.memory_lock=true
      - OPENSEARCH_JAVA_OPTS=-Xms2g -Xmx2g
      - plugins.security.disabled=true
    ulimits:
      memlock:
        soft: -1
        hard: -1
      nofile:
        soft: 65536
        hard: 65536
    volumes:
      - opensearch-data1:/usr/share/opensearch/data
    ports:
      - "9200:9200"
      - "9600:9600"
    networks:
      - rag-network
    healthcheck:
      test: ["CMD-SHELL", "curl -sf http://localhost:9200/_cluster/health || exit 1"]
      interval: 15s
      timeout: 10s
      retries: 8

  opensearch-dashboards:
    image: opensearchproject/opensearch-dashboards:3.6.0
    container_name: opensearch-dashboards
    ports:
      - "5601:5601"
    environment:
      OPENSEARCH_HOSTS: '["http://opensearch-node1:9200"]'
      DISABLE_SECURITY_DASHBOARDS_PLUGIN: "true"
    depends_on:
      opensearch-node1:
        condition: service_healthy
    networks:
      - rag-network

volumes:
  opensearch-data1:

networks:
  rag-network:
    driver: bridge
```

### Passo B — Subir a Stack

```bash
cd ~/mba-rag/infra
docker compose up -d

# Aguardar healthcheck (~60-90 segundos)
docker compose ps

# Verificar resposta
curl http://localhost:9200
# Esperado: {"cluster_name": "opensearch-cluster-rag", ...}
```

**Ubuntu — configurar antes de subir:**
```bash
sudo sysctl -w vm.max_map_count=262144
```

**Windows (PowerShell):**
```powershell
Set-Location "$HOME\mba-rag\infra"
docker compose up -d
Start-Sleep -Seconds 75
Invoke-RestMethod -Uri "http://localhost:9200" | ConvertTo-Json
```

### Passo C — Verificar OpenSearch Dashboards

Acesse no navegador: **http://localhost:5601**  
(Pode levar 2-3 minutos para o Dashboards inicializar após o OpenSearch estar pronto)


---
## PARTE 2 — EXECUCAO NO NOTEBOOK (execute as celulas abaixo)

A partir daqui, todas as células são executadas diretamente no notebook Jupyter local.
O OpenSearch está em `localhost:9200` — sem necessidade de ngrok ou tunelamento.


## CÉLULA 1 — Instalar Dependências e Configurar a URL do OpenSearch

**O que faz:** Instala o cliente Python do OpenSearch e configura a URL local.  
**Por que:** O OpenSearch está rodando localmente via Docker — basta apontar para `localhost:9200`.

In [ ]:
# CÉLULA 1: Instalar dependências e configurar URL local
%pip install opensearch-py==2.7.1 requests --quiet

import os
import requests
import json

# OpenSearch local — sem necessidade de ngrok ou URL externa
OPENSEARCH_URL = os.environ.get('OPENSEARCH_URL', 'http://localhost:9200')

print(f'OpenSearch URL configurada: {OPENSEARCH_URL}')
print('Testando conectividade...')

try:
    r = requests.get(OPENSEARCH_URL, timeout=10)
    if r.status_code == 200:
        info = r.json()
        print(f'Conexao OK!')
        print(f'  Cluster: {info.get("cluster_name", "N/D")}')
        print(f'  Versao:  {info.get("version", {}).get("number", "N/D")}')
    else:
        print(f'OpenSearch respondeu HTTP {r.status_code}')
except requests.exceptions.ConnectionError:
    print('ERRO: OpenSearch nao esta respondendo em localhost:9200')
    print('Solucoes:')
    print('  1. Verifique se o Docker Desktop esta rodando')
    print('  2. Execute no terminal: docker compose up -d')
    print('  3. Aguarde 60-90 segundos e tente novamente')
    print('  Ubuntu: sudo sysctl -w vm.max_map_count=262144')
    raise


### Output Esperado:
```
OpenSearch URL configurada: http://localhost:9200
Testando conectividade...
Conexao OK!
  Cluster: opensearch-cluster-rag
  Versao:  3.6.0
```

### Troubleshooting:
| Situação | Ação |
|----------|------|
| `Connection refused` | Verifique se o Docker Desktop está rodando e execute `docker compose up -d` |
| OpenSearch demora a responder | Aguarde 90 segundos após `docker compose up -d` |
| Ubuntu: container não sobe | Execute `sudo sysctl -w vm.max_map_count=262144` antes de subir |


## CÉLULA 2 — Teste de Conectividade e Saúde do Cluster

**O que faz:** Verifica a saúde do cluster OpenSearch e exibe informações detalhadas.  
**Por que:** Antes de criar índices, precisamos confirmar que o cluster está saudável.

In [ ]:
# Célula 2: Teste de conectividade HTTP via ngrok
import requests
import json
import time

# Headers padrão para requisições ao OpenSearch local
HEADERS = {
    'Content-Type': 'application/json'
}

print('🔌 TESTE DE CONECTIVIDADE — OpenSearch Local')
print('=' * 65)

def testar_conexao(url, tentativas=5, intervalo=5):
    """Testa a conexão com retry automático."""
    for i in range(1, tentativas + 1):
        try:
            r = requests.get(url, headers=HEADERS, timeout=10)
            if r.status_code == 200:
                return r.json()
            else:
                print(f'  Tentativa {i}/{tentativas}: HTTP {r.status_code}')
        except requests.exceptions.ConnectionError as e:
            print(f'  Tentativa {i}/{tentativas}: Conexão recusada — {str(e)[:60]}')
        except requests.exceptions.Timeout:
            print(f'  Tentativa {i}/{tentativas}: Timeout (10s)')
        if i < tentativas:
            time.sleep(intervalo)
    return None

info = testar_conexao(OPENSEARCH_URL)

if info:
    print('\n✅ CONEXÃO ESTABELECIDA!')
    print(f'   Cluster:  {info.get("cluster_name", "N/A")}')
    print(f'   Versão:   {info.get("version", {}).get("number", "N/A")}')
    print(f'   Tagline:  {info.get("tagline", "N/A")}')

    # Verifica saúde do cluster
    health_url = f'{OPENSEARCH_URL}/_cluster/health'
    health = requests.get(health_url, headers=HEADERS).json()
    emoji = {'green': '🟢', 'yellow': '🟡', 'red': '🔴'}.get(health['status'], '⚪')
    print(f'\n{emoji} Cluster status: {health["status"].upper()}')
    print(f'   Nós:          {health["number_of_nodes"]}')
    print(f'   Shards ativos: {health["active_shards"]}')
    if health['status'] == 'yellow':
        print('   ℹ️  YELLOW normal em single-node dev (réplicas não alocadas)')
else:
    print('\n❌ Falha na conexão após 5 tentativas.')
    print('\n🔍 DIAGNÓSTICO — Verifique:')
    print('  1. Docker Desktop está aberto e em execução?')
    print('  2. Contêiner opensearch-node1 está UP? → docker ps')
    print('  3. Túnel ngrok está ativo? → Extensions → ngrok no Docker Desktop')
    print('  4. A URL do ngrok está correta na Célula 1?')
    print('  5. O OpenSearch está saudável? → curl http://localhost:9200 (na sua máquina)')

### Output Esperado:
```
TESTE DE CONECTIVIDADE — OpenSearch Local
=================================================================
CONEXAO ESTABELECIDA!
   Cluster:  opensearch-cluster-rag
   Versao:   3.6.0
   Tagline:  The OpenSearch Project: https://opensearch.org/

Cluster status: YELLOW
   Nos:           1
   Shards ativos: 0
   (Status Yellow e normal em single-node)
```


## 📦 CÉLULA 3 — Criar Cliente Python com opensearch-py

**O que faz:** Instancia o cliente oficial `opensearch-py` apontando para a URL pública do ngrok.

**Por que:** O cliente abstrai as chamadas HTTP, gerencia retries, serialização JSON e será usado em todos os labs subsequentes.

In [ ]:
# CÉLULA 3: Criar cliente opensearch-py local
from opensearchpy import OpenSearch
from urllib.parse import urlparse

# Extrai host e porta da URL pública do ngrok
parsed = urlparse(OPENSEARCH_URL)
ngrok_host = parsed.hostname
# ngrok usa porta 443 (HTTPS) ou 80 (HTTP)
ngrok_port = parsed.port or (443 if parsed.scheme == 'https' else 80)
usa_ssl    = parsed.scheme == 'https'

print(f'🔧 Configurando cliente opensearch-py:')
print(f'   Host:   {ngrok_host}')
print(f'   Porta:  {ngrok_port}')
print(f'   SSL:    {usa_ssl}')

client = OpenSearch(
    hosts=[{'host': ngrok_host, 'port': ngrok_port}],
    http_compress=True,
    use_ssl=usa_ssl,
    verify_certs=False,          # ngrok usa certificado autoassinado
    ssl_assert_hostname=False,
    ssl_show_warn=False,
    http_auth=None,              # Segurança desabilitada no compose de dev
    headers={'ngrok-skip-browser-warning': 'true'}  # Necessário para ngrok gratuito
)

# Teste de conexão via cliente Python
try:
    info = client.info()
    print(f'\n✅ Cliente conectado ao OpenSearch {info["version"]["number"]}')
    print(f'   Cluster: {info["cluster_name"]}')
except Exception as e:
    print(f'\n❌ Erro ao conectar via cliente: {e}')
    print('   Verifique a URL do ngrok e o status do OpenSearch.')

## 📦 CÉLULA 4 — Criar Índice Jurídico

**O que faz:** Cria o índice `documentos_juridicos_aula1` com mapeamento especializado para o domínio jurídico.

**Por que:** O mapeamento define como cada campo é tokenizado e indexado. O `juridico_analyzer` normaliza acentos (`réu → reu`) e remove stopwords, melhorando a precisão das buscas.

In [ ]:
# Célula 4: Criar índice de documentos jurídicos
INDEX_NAME = 'documentos_juridicos_aula1'

# Remove índice anterior se existir (para recriar limpo)
if client.indices.exists(index=INDEX_NAME):
    client.indices.delete(index=INDEX_NAME)
    print(f'🗑️  Índice existente "{INDEX_NAME}" removido')

mapping = {
    "settings": {
        "number_of_shards": 1,
        "number_of_replicas": 0,
        "analysis": {
            "analyzer": {
                "juridico_analyzer": {
                    "type": "custom",
                    "tokenizer": "standard",
                    "filter": [
                        "lowercase",
                        "asciifolding",  # réu → reu, ação → acao
                        "stop"
                    ]
                }
            }
        }
    },
    "mappings": {
        "properties": {
            "titulo":          {"type": "text",    "analyzer": "juridico_analyzer",
                                "fields": {"keyword": {"type": "keyword"}}},
            "conteudo":        {"type": "text",    "analyzer": "juridico_analyzer"},
            "tipo_documento":  {"type": "keyword"},
            "tribunal":        {"type": "keyword"},
            "numero_processo": {"type": "keyword"},
            "data_julgamento": {"type": "date"},
            "relator":         {"type": "keyword"}
        }
    }
}

resp = client.indices.create(index=INDEX_NAME, body=mapping)
if resp.get('acknowledged'):
    print(f'✅ Índice "{INDEX_NAME}" criado no OpenSearch local!')
    print('   Campos: titulo · conteudo · tipo_documento · tribunal')
    print('           numero_processo · data_julgamento · relator')
else:
    print('❌ Falha ao criar índice:', resp)

## 📦 CÉLULA 5 — Inserir Documentos Jurídicos de Exemplo

**O que faz:** Indexa 5 documentos jurídicos fictícios enviados do Colab para o OpenSearch local via túnel ngrok.

**Por que:** Demonstra que a pipeline de escrita remota funciona — os dados ficam na sua máquina, não no Colab.

In [ ]:
# Célula 5: Indexar documentos jurídicos
import time

documentos = [
    {
        "titulo": "Acórdão — Peculato Doloso — Servidor Público da Saúde",
        "conteudo": (
            "EMENTA: PENAL. PECULATO DOLOSO. SERVIDOR PÚBLICO. DESVIO DE VERBAS DO SUS. "
            "MATERIALIDADE E AUTORIA COMPROVADAS. O réu, servidor do Ministério da Saúde, "
            "desviou R$ 2.500.000,00 destinados a equipamentos médicos, configurando o "
            "crime de peculato doloso (art. 312 CP). Recurso desprovido."
        ),
        "tipo_documento": "acordao",
        "tribunal": "STJ",
        "numero_processo": "REsp 1.987.654/SP",
        "data_julgamento": "2024-03-15",
        "relator": "Min. João Silva"
    },
    {
        "titulo": "Habeas Corpus — Prisão Preventiva — Tráfico de Drogas",
        "conteudo": (
            "EMENTA: HABEAS CORPUS. TRÁFICO DE DROGAS. PRISÃO PREVENTIVA SEM "
            "FUNDAMENTAÇÃO IDÔNEA. EXCESSO DE PRAZO. CONSTRANGIMENTO ILEGAL. ORDEM CONCEDIDA. "
            "Meras referências abstratas à gravidade do delito não bastam para justificar "
            "a custódia cautelar. Medidas alternativas impostas."
        ),
        "tipo_documento": "acordao",
        "tribunal": "STF",
        "numero_processo": "HC 198.432/RJ",
        "data_julgamento": "2024-06-20",
        "relator": "Min. Maria Santos"
    },
    {
        "titulo": "Boletim de Ocorrência — Furto Qualificado — Escalada",
        "conteudo": (
            "BO Nº 2024/0045678. Furto Qualificado por escalada. Vítima retornou ao domicílio "
            "e encontrou janela arrombada. Bens subtraídos: 1 notebook, 2 celulares, "
            "R$1.200,00. Impressões digitais coletadas. Câmera de segurança registrou suspeito "
            "encapuzado. Inquérito instaurado pela delegada responsável."
        ),
        "tipo_documento": "boletim_ocorrencia",
        "tribunal": "PCSP",
        "numero_processo": "BO 2024/0045678",
        "data_julgamento": "2024-11-12",
        "relator": "Del. Carlos Mendes"
    },
    {
        "titulo": "Laudo Pericial — Homicídio — Análise Balística",
        "conteudo": (
            "LAUDO Nº 2024/LP-00234. Análise balística: 3 estojos calibre 9mm da mesma arma "
            "(análise de estrias). Trajetória compatível com vítima ajoelhada. "
            "GSR (Gunshot Residue) encontrado nas mãos do suspeito. "
            "Conclusão: elementos compatíveis com homicídio qualificado por motivo torpe."
        ),
        "tipo_documento": "laudo_pericial",
        "tribunal": "IML-SP",
        "numero_processo": "IP 2024/00789",
        "data_julgamento": "2024-09-08",
        "relator": "Perita Dra. Ana Lima"
    },
    {
        "titulo": "Sentença — Absolvição por Legítima Defesa — Homicídio",
        "conteudo": (
            "SENTENÇA ABSOLUTÓRIA. Processo: 0001234-56.2024.8.26.0001. "
            "O réu agiu em legítima defesa própria, com uso moderado dos meios "
            "necessários para repelir injusta agressão atual (vítima avançou com faca). "
            "Testemunhas oculares confirmaram a versão defensiva. "
            "Fundamento: art. 25 CP c/c art. 386 VI CPP. Réu absolvido."
        ),
        "tipo_documento": "sentenca",
        "tribunal": "TJ-SP",
        "numero_processo": "0001234-56.2024.8.26.0001",
        "data_julgamento": "2024-12-01",
        "relator": "Juíza Dra. Paula Costa"
    }
]

print(f'📥 Indexando {len(documentos)} documentos no OpenSearch local via ngrok...')
print(f'   Destino: {OPENSEARCH_PUBLIC_URL}/{INDEX_NAME}')
print('=' * 65)

sucesso = 0
for i, doc in enumerate(documentos, 1):
    try:
        resp = client.index(
            index=INDEX_NAME,
            body=doc,
            id=str(i),
            refresh=True          # Índice atualizado imediatamente
        )
        ok = resp['result'] in ['created', 'updated']
        print(f'  {"✅" if ok else "❌"} [{i}] {doc["titulo"][:55]}...')
        if ok:
            sucesso += 1
    except Exception as e:
        print(f'  ❌ [{i}] Erro: {str(e)[:80]}')

total = client.count(index=INDEX_NAME)['count']
print(f'\n✅ {sucesso}/{len(documentos)} documentos indexados | Total no índice: {total}')

## CÉLULA 6 — Validação: Executar Buscas no OpenSearch

**O que faz:** Executa 3 buscas diferentes para validar o índice criado.  
**Por que:** Confirma que os documentos foram indexados corretamente e que a busca por texto funciona.

In [ ]:
# CÉLULA 6: Executar buscas de validação

def buscar(query_text, filtro_tipo=None, top_k=3):
    """Busca full-text no OpenSearch com filtro opcional por tipo de documento."""
    query = {
        "query": {
            "bool": {
                "must": [{
                    "multi_match": {
                        "query": query_text,
                        "fields": ["titulo^2", "conteudo"],
                        "type": "best_fields"
                    }
                }],
                "filter": ([{"term": {"tipo_documento": filtro_tipo}}]
                           if filtro_tipo else [])
            }
        },
        "size": top_k,
        "_source": ["titulo", "tipo_documento", "tribunal", "numero_processo"]
    }
    return client.search(index=INDEX_NAME, body=query)


print('🔍 BUSCAS DE VALIDAÇÃO — Notebook → OpenSearch Local')
print('=' * 65)

casos_teste = [
    ('peculato servidor público desvio verba', None),
    ('homicídio balística projéteis arma', 'laudo_pericial'),
    ('prisão preventiva fundamentação cautelar', None),
]

todos_ok = True
for query, filtro in casos_teste:
    print(f'\n  🔎 Query: "{query}"' + (f'  [filtro: {filtro}]' if filtro else ''))
    try:
        hits = buscar(query, filtro_tipo=filtro)['hits']['hits']
        if hits:
            for h in hits:
                src = h['_source']
                print(f'     📄 [{h["_score"]:.2f}] {src["titulo"][:55]}...')
                print(f'          {src["tipo_documento"]} | {src["tribunal"]}')
        else:
            print('     ⚠️  Nenhum resultado retornado')
            todos_ok = False
    except Exception as e:
        print(f'     ❌ Erro na busca: {e}')
        todos_ok = False

print('\n' + '=' * 65)
if todos_ok:
    print('✅ TODAS AS BUSCAS RETORNARAM RESULTADOS RELEVANTES!')
    print('   Pipeline completo validado: Notebook → OpenSearch → resposta')
else:
    print('⚠️  Algumas buscas não retornaram resultados. Verifique a indexação (Célula 5).')

### Output Esperado:
```
BUSCAS DE VALIDAÇÃO — OpenSearch Local
================================================================
Busca 1: homicidio
  OK: 1 resultado(s)
  1. Acórdão - Homicídio Qualificado (score: 2.34)

Busca 2: habeas corpus prisao
  OK: 2 resultado(s)
  1. Habeas Corpus - Excesso de Prazo (score: 3.21)
  ...
```


## 📦 CÉLULA 7 — Script de Validação Completa (Checklist Automatizado)

**O que faz:** Executa automaticamente todos os 8 itens do checklist de validação do Lab 1.

**Por que:** Fornece uma verificação objetiva e reproduzível do ambiente para fins de avaliação.

In [ ]:
# Célula 7: Script de validação completa — Checklist automatizado
from datetime import datetime

checklist = []

def check(nome, fn):
    try:
        resultado = fn()
        checklist.append(('✅', nome, str(resultado)[:60] if resultado else 'OK'))
    except Exception as e:
        checklist.append(('❌', nome, str(e)[:60]))

print(f'📋 CHECKLIST DO LAB 1 — {datetime.now().strftime("%d/%m/%Y %H:%M")}')
print('=' * 65)

# 1. Biblioteca opensearch-py importada
def c1():
    from opensearchpy import OpenSearch
    return 'opensearch-py importado'
check('opensearch-py instalado', c1)

# 2. URL localhost configurada
def c2():
    assert 'localhost' in OPENSEARCH_URL or OPENSEARCH_URL.startswith('https://'), \
        'URL não parece ser localhost'
    return OPENSEARCH_URL
check('URL localhost configurada', c2)

# 3. OpenSearch acessível via localhost
def c3():
    r = requests.get(OPENSEARCH_URL, headers=HEADERS_NGROK, timeout=10)
    assert r.status_code == 200
    return f'versão {r.json()["version"]["number"]}'
check('OpenSearch acessível via localhost', c3)

# 4. Cliente Python conectado
def c4():
    info = client.info()
    return f'cluster {info["cluster_name"]}'
check('Cliente opensearch-py conectado', c4)

# 5. Cluster saudável (green ou yellow)
def c5():
    h = requests.get(f'{OPENSEARCH_URL}/_cluster/health',
                     headers=HEADERS_NGROK).json()
    assert h['status'] in ['green', 'yellow'], f'Status: {h["status"]}'
    return f'status {h["status"]}'
check('Cluster saudável (green/yellow)', c5)

# 6. Índice criado
def c6():
    assert client.indices.exists(index=INDEX_NAME), 'Índice não encontrado'
    return f'índice {INDEX_NAME} existe'
check(f'Índice "{INDEX_NAME}" criado', c6)

# 7. Documentos indexados
def c7():
    cnt = client.count(index=INDEX_NAME)['count']
    assert cnt >= 5, f'Apenas {cnt} documentos'
    return f'{cnt} documentos'
check('5+ documentos indexados', c7)

# 8. Busca retorna resultados relevantes
def c8():
    hits = buscar('peculato servidor público')['hits']['hits']
    assert len(hits) > 0, 'Nenhum resultado'
    top = hits[0]['_source']['titulo']
    return f'top: {top[:40]}'
check('Busca retorna resultados', c8)

# Exibe checklist
print()
for status, nome, detalhe in checklist:
    print(f'  {status} {nome}')
    if status == '❌':
        print(f'     └─ {detalhe}')

ok = sum(1 for s, _, _ in checklist if s == '✅')
total = len(checklist)
print(f'\n  Pontuação: {ok}/{total}')

if ok == total:
    print('\n🎉 LAB 1 COMPLETAMENTE VALIDADO!')
    print('   Pipeline Notebook → localhost → OpenSearch Local funcionando!')
else:
    print(f'\n⚠️  {total-ok} item(ns) com falha. Consulte o troubleshooting acima.')

## Checklist de Validação do Lab 1

| # | Item | Verificação |
|---|------|-------------|
| 1 | Docker Desktop rodando | `docker ps` mostra opensearch-node1 |
| 2 | OpenSearch respondendo | `curl http://localhost:9200` retorna JSON |
| 3 | Cluster status yellow/green | `/_cluster/health` confirma |
| 4 | Índice criado | `documentos_juridicos_aula1` existe |
| 5 | Documentos indexados | 5 documentos inseridos |
| 6 | Buscas funcionando | Resultados relevantes retornados |

**Pontuação:** 6/6 = Lab 1 completo

## Próximo Passo
Continue para o **LAB2 — Setup Ambiente Local** para verificar Python, Ollama e as demais dependências.
